# Reshaping: format długi ↔ szeroki w pandas i polars

**Problem:** "reshaping" to szerokie pojęcie — zmiana KSZTAŁTU danych, nie ich treści ani agregacji. pandas ma na to kilka osobnych narzędzi (`melt`, `wide_to_long`, `stack`/`unstack`, `explode`), które łatwo pomylić z `groupby` i `pivot_table`, mimo że robią coś fundamentalnie innego.

**Format długi (long) vs szeroki (wide):**
- **Długi** — jeden wiersz = jedna obserwacja, kategoria opisana jako WARTOŚĆ w kolumnie (np. `region, year, sales`).
- **Szeroki** — jedna kategoria = osobna KOLUMNA (np. `region, sales_2024, sales_2025`).

**Czym reshaping różni się od `groupby` i `pivot` (mają osobne, głębsze notatki w tym repo):**
- `groupby` **redukuje liczbę wierszy** przez agregację (wiele wierszy → jeden na grupę). To zmiana treści, nie tylko kształtu.
- `pivot`/`pivot_table` to reshaping long→wide **połączony z agregacją** — jeśli kombinacje indeks+kolumna się powtarzają, trzeba je czymś połączyć (`aggfunc`).
- **Czysty reshaping** (`melt`, `stack`/`unstack`, `wide_to_long`) **nie agreguje niczego** — tylko przekłada te same dane z jednego układu na drugi. Liczba "komórek z danymi" się nie zmienia, zmienia się tylko ich układ.

**Kiedy stosować co:** `melt`/`stack` gdy idziesz wide→long; `pivot`/`unstack` gdy long→wide (i wiesz, że kombinacje są unikalne — bez agregacji, patrz notatka o pivotowaniu); `wide_to_long` gdy kolumny mają wspólny prefiks + zmienny sufiks (np. lata); `explode` gdy "szerokość" siedzi nie w kolumnach, tylko w liście wewnątrz pojedynczej komórki.

## Setup

In [ ]:
import pandas as pd
import polars as pl
import numpy as np

# Format szeroki: kolumny z sufiksem roku - typowy przypadek pod melt / wide_to_long
wide = pd.DataFrame({
    "region": ["North", "South", "East", "West"],
    "sales_2024": [1200, 900, 700, 1300],
    "sales_2025": [1350, 950, 720, 1400],
    "units_2024": [12, 9, 7, 13],
    "units_2025": [14, 10, 8, 15],
})
wide

## Sekcja 1 — `melt()`: format szeroki → długi

- `id_vars` — kolumny, które ZOSTAJĄ jako identyfikator (nie są "topione").
- `value_vars` — które kolumny stopić (domyślnie: wszystkie poza `id_vars`).
- `var_name` / `value_name` — nazwy dwóch nowych kolumn wynikowych.

In [ ]:
wide.melt(id_vars="region", var_name="metric_year", value_name="value")

In [ ]:
# value_vars - stopienie tylko wybranych kolumn (tu: pomijamy 'units_*')
wide.melt(id_vars="region", value_vars=["sales_2024", "sales_2025"], var_name="year_col", value_name="sales")

## Sekcja 2 — `wide_to_long()`: gdy kolumny mają wspólny prefiks + zmienny sufiks

`melt` topi każdą kolumnę do jednej wspólnej kolumny `value` — traci rozróżnienie między `sales` a `units`. `wide_to_long()` jest do tego stworzone: `stubnames` to lista prefiksów (`sales`, `units`), `i` to kolumna identyfikująca wiersz, `j` to nazwa nowej kolumny z sufiksem (tu: rok). Wynik ma osobną kolumnę `sales` I osobną `units` — nie jedną zlepioną `value`.

In [ ]:
pd.wide_to_long(wide, stubnames=["sales", "units"], i="region", j="year", sep="_").reset_index()

## Sekcja 3 — `stack()` / `unstack()`: reshaping przez poziomy `MultiIndex`

Gdy dane mają hierarchiczne kolumny (`MultiIndex`, np. `(metric, year)`), `stack()` przenosi wskazany poziom kolumn do indeksu (szeroki → długi), `unstack()` robi odwrotnie. `level=` wybiera, KTÓRY poziom przenieść, gdy jest ich więcej niż jeden.

In [ ]:
wide_multi = wide.set_index("region")
wide_multi.columns = pd.MultiIndex.from_tuples(
    [("sales", "2024"), ("sales", "2025"), ("units", "2024"), ("units", "2025")]
)
wide_multi

In [ ]:
# domyślnie: OSTATNI poziom kolumn (tu: rok) trafia do indeksu
wide_multi.stack()

In [ ]:
# level=0 - przenosimy KONKRETNY poziom (tu: metric, zamiast domyślnego ostatniego)
wide_multi.stack(level=0)

**Warto wiedzieć:** w aktualnych wersjach pandas `stack()` domyślnie **zachowuje** `NaN` w wyniku (nie usuwa brakujących kombinacji tak, jak robiła to starsza implementacja sprzed pandas 2.1). Jeśli pracujesz na starszym środowisku i widzisz inne zachowanie niż tutaj, sprawdź wersję pandas.

## Sekcja 4 — `explode()`: lista w komórce → wiele wierszy

To inny rodzaj "szerokości" niż kolumny — dane mogą być "szerokie" wewnątrz pojedynczej komórki (lista tagów, kilka ID w jednym polu). `explode()` rozbija taką listę na osobne wiersze, powielając pozostałe kolumny.

In [ ]:
orders = pd.DataFrame({
    "order_id": [1, 2, 3],
    "region": ["North", "South", "East"],
    "tags": [["electronics", "sale"], ["cables"], ["wifi", "new", "sale"]],
})
print(orders)
print()
orders.explode("tags")

## Sekcja 5 — `get_dummies()`: kategorie jako osobne kolumny (one-hot)

Odwrotność "długiej kategorii" — każda unikalna wartość dostaje własną kolumnę `True`/`False`. Częste jako krok przygotowawczy pod modele ML albo pod macierz korelacji zmiennych kategorycznych.

In [ ]:
pd.get_dummies(wide["region"])

## Sekcja 6 — Transpozycja: `.T`

Najprostszy "reshaping" — zamienia wiersze z kolumnami miejscami. Rzadko przydatne do analizy, częściej do szybkiego podglądu wąskiej, ale wysokiej tabeli (np. pojedynczy wiersz z wieloma metrykami) w bardziej czytelnym układzie.

In [ ]:
wide.set_index("region").loc[["North", "South"]].T

## Sekcja 7 — polars: `explode()`

polars ma bezpośredni odpowiednik `explode()` o tej samej logice. `melt`/`pivot`/`unpivot` w polars mają już osobną, głębszą notatkę w tym repo — tu tylko `explode`, bo nigdzie indziej się nie pojawił.

In [ ]:
orders_pl = pl.DataFrame({
    "order_id": [1, 2, 3],
    "region": ["North", "South", "East"],
    "tags": [["electronics", "sale"], ["cables"], ["wifi", "new", "sale"]],
})
orders_pl.explode("tags")

## Sekcja 8 — Pułapki

### Pułapka 1 — `melt()` bez `id_vars` topi DOSŁOWNIE wszystko, łącznie z kolumnami identyfikującymi

Bez jawnego `id_vars`, `melt()` nie ma pojęcia, że `region` miała zostać jako identyfikator — traktuje ją jak każdą inną kolumnę do stopienia. Efekt: nazwy regionów lądują wymieszane z liczbami sprzedaży w tej samej kolumnie `value`, jako tekst.

In [ ]:
wide.melt()  # brak id_vars - 'region' też zostaje stopiona

### Pułapka 2 — `wide_to_long()` z niedopasowanym `sep` daje PUSTY wynik, nie błąd

Jeśli separator w nazwach kolumn (np. `sales-2024`, myślnik) nie zgadza się z parametrem `sep` przekazanym do funkcji (`sep="_"`), `wide_to_long()` **nie znajduje żadnego dopasowania** — i zamiast błędu zwraca pusty `DataFrame`. To najbardziej podstępna pułapka w tej notatce: kod się wykonuje, nic nie krzyczy, a wynik jest po prostu pusty.

In [ ]:
wide_dash = wide.rename(columns={c: c.replace("_", "-") for c in wide.columns if c != "region"})
print(f"Nazwy kolumn: {wide_dash.columns.tolist()}\n")

# sep="_" nie pasuje do rzeczywistego separatora "-" w nazwach kolumn
result = pd.wide_to_long(wide_dash, stubnames=["sales", "units"], i="region", j="year", sep="_")
print(f"Liczba wierszy wyniku: {len(result)}  <- PUSTO, bez żadnego błędu")
result

### Pułapka 3 — `explode()`: pusta lista `[]` i `NaN` dają IDENTYCZNY wynik

"Wiersz z pustą listą tagów" i "wiersz, gdzie danych o tagach w ogóle brakuje" to semantycznie różne sytuacje — ale po `explode()` obie stają się nie do odróżnienia: obie zamieniają się w pojedynczy wiersz z `NaN`. Jeśli to rozróżnienie ma znaczenie, trzeba je sprawdzić PRZED `explode()`, nie po.

In [ ]:
orders_edge = pd.DataFrame({
    "order_id": [1, 2, 3],
    "tags": [["a", "b"], [], np.nan],  # order_id 2: pusta lista; order_id 3: brak danych
})
print(orders_edge)
print()
orders_edge.explode("tags")  # wiersze 2 i 3 wyglądają identycznie w wyniku

## Podsumowanie

| Sytuacja | Narzędzie |
|---|---|
| Szeroki → długi, jedna wspólna kolumna wartości | `melt(id_vars=, value_vars=, var_name=, value_name=)` |
| Szeroki → długi, kolumny mają wspólny prefiks + zmienny sufiks (np. rok) | `wide_to_long(stubnames=, i=, j=, sep=)` |
| Szeroki → długi / długi → szeroki przez poziomy `MultiIndex` | `stack(level=)` / `unstack(level=)` |
| Lista w komórce → osobne wiersze | `explode("kolumna")` (pandas i polars) |
| Kategorie jako osobne kolumny 0/1 | `get_dummies()` |
| Wiersze ↔ kolumny (dosłowna zamiana) | `.T` |
| Długi → szeroki, kombinacje unikalne (bez agregacji) | `pivot()` — patrz notatka o pivotowaniu |
| Długi → szeroki, kombinacje się powtarzają (z agregacją) | `pivot_table()` — patrz notatka o pivotowaniu |
| Redukcja liczby wierszy przez agregację | `groupby()` — patrz notatka o grupowaniu |

**Wniosek:** ten sam motyw co w pozostałych notatkach — najgroźniejsze pułapki reshapingu (`wide_to_long` z pustym wynikiem, `melt` bez `id_vars`, `explode` maskujące różnicę między "pusto" a "brak danych") nie rzucają błędu. Przy każdej operacji reshapingu warto porównać liczbę wierszy/kolumn przed i po — to najtańszy sposób złapania problemu, zanim trafi dalej w analizę.